<div align="center">

# NASA_Turbofan_RUL

</div>

L'objectif de ce projet est de prédire la durée de vie restante des turbo-réacteurs en fonction de paramètres tels que le régime moteur, la pression et la température. Je vais effectuer une analyse exploratoire des données (EDA) pour comprendre les relations entre les variables. Ensuite, je développerai un modèle prédictif pour estimer la RUL (Remaining Useful Life). L'objectif est d'obtenir une erreur raisonnable de maximum 20 cycles sur le jeu de données FD001 en testant différents modèles, tout en quantifiant l'incertitude associée à cette estimation à l'aide d'un intervalle de confiance.

<h2>Index</h2>

* [**Chargement des données**](#data-load)
* [**Analyse Exploratoire des Données (EDA)**](#eda)
    * [**Analyse fondamentale**](#analyse-fondamentale)
    * [**Les variables**](#variables)
    * [**Analyse univariée**](#analyse-univariee)
    * [**Analyse multivariée**](#analyse-multivariee) 
* [**Tests statistiques**](#test-statistiques)
* [**Pre-processing**](#pre-processing)
* [**Modélisation**](#modelisation)
* [**Performances du modèle**](#performances-modele)

<h2>Import</h2>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

<a id="data-load"></a>
<h2>Chargement des données</h2>

In [3]:
col_names = ["unit_number", "time_cycles", "setting_1", "setting_2", "setting_3"]
col_names += [f'sensor_{i}' for i in range(1, 22)]

In [4]:
df_train = pd.read_csv("data/CMaps/train_FD001.txt", sep=r"\s+", header=None, names=col_names, index_col=False)
df_test = pd.read_csv("data/CMaps/test_FD001.txt", sep=r"\s+", header=None, names=col_names, index_col=False)

In [5]:
df_train.head()

,unit_number,time_cycles,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [6]:
df_test.head()

,unit_number,time_cycles,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,521.72,2388.03,8125.55,8.4052,0.03,392,2388,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,522.16,2388.06,8139.62,8.3803,0.03,393,2388,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,521.97,2388.03,8130.10,8.4441,0.03,393,2388,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,521.38,2388.05,8132.90,8.3917,0.03,391,2388,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,522.15,2388.03,8129.54,8.4031,0.03,390,2388,100.0,38.99,23.4130


In [7]:
df_train.shape

(20631, 26)

<a id="eda"></a>

## Analyse Exploratoire des Données (EDA)

<a id="analyse-fondamentale"></a>

### **Analyse fondamentale**

**Quoi** :

Le dataset est composé de 4 sous-ensembles (FD001 à FD004) simulant le comportement de turboréacteurs. Chaque ensemble combine des scénarios distincts :

   - Conditions opérationnelles : Soit stables (niveau de la mer), soit variées (6 conditions de vol simulant différentes altitudes et vitesses).

   - Modes de défaillance : Dégradation du compresseur haute pression (HPC) seule, ou combinée à une dégradation de la soufflante (Fan).

   - Structure : Chaque fichier contient 26 colonnes (ID moteur, Cycles, 3 réglages pilotes et 21 capteurs mesurant pressions, températures et vitesses de rotation).

**Quand** & **Qui** :

Les données ont été produites en 2008 par l'équipe de pronostic de la NASA Ames Research Center, notamment par A. Saxena et K. Goebel. Elles ont servi de base au "PHM 2008 Data Challenge", devenu depuis la référence mondiale de la maintenance prédictive.

**Comment** :

Les données ne sont pas issues de vrais vols, mais d'un simulateur haute fidélité nommé C-MAPSS (Commercial Modular Aero-Propulsion System Simulation).

   - Le principe : On injecte une dégradation initiale invisible qui croît de manière exponentielle ou linéaire jusqu'à ce que le moteur atteigne un seuil critique (la panne).

   - Le bruit : Un bruit de mesure "réaliste" est ajouté aux capteurs pour complexifier la prédiction.

**Pourquoi** :

L'enjeu est l'estimation de la RUL (Remaining Useful Life), soit la "Durée de Vie Utile Restante".

   - Objectif : Prédire exactement combien de cycles de vol il reste à un moteur avant la défaillance.

   - Impact : Passer d'une maintenance curative (on répare quand c'est cassé) ou préventive (on change trop tôt par sécurité) à une maintenance prédictive (on change juste avant la panne pour optimiser les coûts et la sécurité).